# Data Joining Pipeline (Revised)

For a detailed log of pipeline revisions, see `SourceOfTruth.md#5-pipeline-revision-log`.


In [1]:
import pandas as pd
import numpy as np
import gc

In [2]:
# Load the cleaned outputs from DataCleaning-Rev.ipynb
# These files include the audit trail column 'original_amount_header'
df_Trans = pd.read_parquet('transactions_capping.parquet')
df_TransItem = pd.read_parquet('transaction_items_cleaned.parquet')
df_Users = pd.read_parquet('users_cleaned.parquet')
df_menu = pd.read_parquet('menu_cleaned.parquet')
df_stores = pd.read_parquet('stores_cleaned.parquet')

In [3]:
# VALIDATE: Verify the audit trail column exists (DataCleaning-Rev enhancement)
required_cols = ['original_amount_header', 'original_amount', 'discount_applied', 'final_amount']
missing = [c for c in required_cols if c not in df_Trans.columns]
if missing:
    print(f"WARNING: Missing required columns in df_Trans: {missing}")
    print("This notebook expects outputs from DataCleaning-Rev.ipynb which includes original_amount_header.")
    gc.collect()
else:
    print(f"All {len(required_cols)} required columns present in df_Trans.")
    print(f"original_amount_header sample values:\n{df_Trans['original_amount_header'].head(3)}")
    print(f"original_amount (corrected) sample values:\n{df_Trans['original_amount'].head(3)}")
    gc.collect()

gc.collect()
# Validate item-level data
print(f"\ndf_Trans: {len(df_Trans):,} transactions")
print(f"df_TransItem: {len(df_TransItem):,} item lines")
print(f"df_Users: {len(df_Users):,} users")

All 4 required columns present in df_Trans.
original_amount_header sample values:
0    38.0
1    33.0
2    27.0
Name: original_amount_header, dtype: float32
original_amount (corrected) sample values:
0    28.5
1    33.0
2    27.0
Name: original_amount, dtype: float32

df_Trans: 14,623,691 transactions
df_TransItem: 26,885,688 item lines
df_Users: 2,196,257 users


In [4]:
gc.collect()

0

# Data Master Construction

Build a fully-denormalized item-level master table via left merges: transaction headers with user/store info, then item details with menu catalog.

In [5]:
# Step 1: Build transaction-level master (header + users + stores)
gc.collect()
df_MasterTrans = (
    df_Trans
    .merge(df_Users, on='user_id', how='left')
    .merge(df_stores, on='store_id', how='left')
)

In [6]:
gc.collect()

0

In [7]:
# Step 2: Build item-level master (items + menu + transaction_header + users + stores)
gc.collect()
df_Master = (
    df_TransItem
    .merge(df_menu, on='item_id', how='left')
    .merge(
        df_MasterTrans,
        on='transaction_id',
        how='left'
    )
)

In [8]:
gc.collect()

0

In [9]:
# Report row counts for verification
print(f"df_MasterTrans (transaction level): {len(df_MasterTrans):,} rows")
print(f"df_Master (item level):            {len(df_Master):,} rows")
print(f"Expected items (from df_TransItem): {len(df_TransItem):,} rows")

if len(df_Master) == len(df_TransItem):
    gc.collect()
    print("\n[OK] Row count match: All items preserved through joins")
else:
    diff = len(df_Master) - len(df_TransItem)
    gc.collect()
    print(f"\n[WARN] Row count mismatch: {diff:+d} rows (possible duplicate/missing transactions)")

df_MasterTrans (transaction level): 14,623,691 rows
df_Master (item level):            26,885,688 rows
Expected items (from df_TransItem): 26,885,688 rows

[OK] Row count match: All items preserved through joins


In [10]:
gc.collect()

0

## Validation

In [11]:
def validate_and_clean_master(df_master, df_header, df_item_raw):
    """
    Validates the joined master table and fixes column conflicts.
    
    Checks:
    - Row count integrity vs item-level input
    - Orphan records (items without menu/store/header)
    - Financial audit (item subtotals match header original_amount)
    - Cleans up redundant _x / _y column suffixes
    - Verifies original_amount_header audit trail is intact
    """
    print("=== MASTER VALIDATION ===\n")

    # 1. Row Count Integrity
    row_diff = len(df_master) - len(df_item_raw)
    if row_diff == 0:
        gc.collect()
        print(f"Row Count Check: OK ({len(df_master):,} rows)")
    else:
        gc.collect()
        print(f"Row Count Check: Mismatch ({row_diff:,} row difference)")
    gc.collect()

    # 2. Orphan Check
    print("\nOrphan Check:")
    orphans = {
        'Menu': df_master['item_name'].isnull().sum(),
        'Store': df_master['store_name'].isnull().sum(),
        'Header': df_master['original_amount'].isnull().sum()
    }
    for key, val in orphans.items():
        print(f"- {key}: {val:,} orphan records")
    gc.collect()

    # 3. Financial Audit: Verify original_amount = sum(item subtotals)
    print("\nFinancial Audit:")
    audit_items = (
        df_master
        .groupby('transaction_id')['subtotal']
        .sum()
        .reset_index()
    )
    audit_merge = audit_items.merge(
        df_header[['transaction_id', 'original_amount']],
        on='transaction_id',
        how='inner'
    )
    audit_merge['diff'] = (
        audit_merge['original_amount'] -
        audit_merge['subtotal']
    ).abs()
    mismatches = audit_merge[audit_merge['diff'] > 0.01]
    if len(mismatches) == 0:
        gc.collect()
        print(f"Balanced Transactions: {len(audit_merge):,} [OK]")
    else:
        gc.collect()
        print(f"Mismatched Transactions: {len(mismatches):,} [WARN]")
    gc.collect()

    # 4. Verify original_amount_header audit trail
    print("\nAudit Trail Check:")
    if 'original_amount_header' in df_master.columns:
        header_count = df_master['original_amount_header'].notna().sum()
        print(f"original_amount_header: {header_count:,} non-null values (preserved [OK])")
        gc.collect()
        # Check that at least some headers differ from corrected amounts
        diff_count = (df_master['original_amount_header'] != df_master['original_amount']).sum()
        print(f"Transactions with header ≠ corrected: {diff_count:,} ({100*diff_count/len(df_master):.1f}%)")
    else:
        print("WARNING: original_amount_header not found! Audit trail missing.")
    gc.collect()

    # 5. Remove Redundant Columns (using explicit list for safety)
    print("\nColumn Cleanup:")
    # Identify _x / _y suffixed columns
    gc.collect()
    redundant_cols = [
        col for col in df_master.columns
        if col.endswith('_x') or col.endswith('_y')
    ]
    
    # Merge duplicate created_at columns: prefer _x (from transaction header)
    if 'created_at_x' in df_master.columns:
        gc.collect()
        df_master['created_at'] = df_master['created_at_x']
        print("Merged created_at_x → created_at")
    
    if redundant_cols:
        gc.collect()
        df_master.drop(
            columns=redundant_cols,
            inplace=True,
            errors='ignore'
        )
        print(f"Removed {len(redundant_cols)} redundant columns: {', '.join(redundant_cols)}")
    else:
        print("No redundant columns found")

    print("\n=== VALIDATION COMPLETED ===")
    return df_master


# Execute validation
df_Master = validate_and_clean_master(
    df_Master,
    df_MasterTrans,
    df_TransItem
)
gc.collect()

=== MASTER VALIDATION ===

Row Count Check: OK (26,885,688 rows)

Orphan Check:
- Menu: 0 orphan records
- Store: 0 orphan records
- Header: 0 orphan records

Financial Audit:
Balanced Transactions: 14,623,691 [OK]

Audit Trail Check:
original_amount_header: 26,885,688 non-null values (preserved [OK])
Transactions with header ≠ corrected: 3,883,257 (14.4%)

Column Cleanup:
Merged created_at_x → created_at
Removed 2 redundant columns: created_at_x, created_at_y

=== VALIDATION COMPLETED ===


0

In [12]:
gc.collect()

0

In [13]:
# Merge payment method metadata
df_payment = pd.read_parquet('paymentsUnique.parquet')
gc.collect()
df_Master = df_Master.merge(
    df_payment,
    left_on='payment_method_id',
    right_on='method_id',
    how='left'
)

In [14]:
gc.collect()

0

In [15]:
# Save final master table for downstream notebooks
df_Master.to_parquet('df_Master_Final.parquet', index=False)
print(f"Saved df_Master_Final.parquet: {len(df_Master):,} rows, {len(df_Master.columns)} columns")

Saved df_Master_Final.parquet: 26,885,688 rows, 39 columns


In [16]:
# Free memory
del df_Trans, df_TransItem, df_Users, df_menu, df_stores, df_MasterTrans
gc.collect()

0